The anatomical annotations (also referred to as coordinates) in our dataset were performed by two orthodontists. Here, we present an analysis to assess the reliability and inter-annotator agreement of their annotations, primarily using ANOVA and the Intraclass Correlation Coefficient (ICC).

In [1]:
library(yaml)
library(dplyr)
library(tibble)
library(tidyr)
library(psych) # ICC computation
library(tidyverse)

source("../scripts/procrustes_analysis.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ purrr     1.0.4
✔ ggplot2   3.5.2     ✔ readr     2.1.5
✔ lubridate 1.9.4     ✔ stringr   1.5.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ ggplot2::%+%()   masks psych::%+%()
✖ ggplot2::alpha() masks psych::alpha()
✖ dplyr::filter()  masks stats::filter()
✖ dplyr::lag()     masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘shapes’


The following object is masked from ‘package:psych’:

    tr




In [2]:
# Load the configuration file
config <- yaml::yaml.load_file("../config.yml")

In [3]:
# Load dataset with replicate coordinates annotations (from 2 operators)
df_replicate_coordinates <- read.csv(paste0("../", config$dataset$coord_annotation_reliability))
df_replicate_coordinates$Operator <- plyr::mapvalues(df_replicate_coordinates$Operator,
                                                     from = c("CT", "AS"),
                                                     to = c("M.C.T", "A.I.L"))
df_replicate_coordinates$Patient_ID <- paste(df_replicate_coordinates$Patient, df_replicate_coordinates$Operator, sep = "-")

In [4]:
output_path <- "../outputs/annotation_reliability/"
# Create the output directory if it does not exist
if (!dir.exists(output_path)) {
  dir.create(output_path, recursive = TRUE)
}

In [5]:
head(df_replicate_coordinates)

,Patient,Operator,X_Sella,Y_Sella,X_Basion,Y_Basion,X_PNS,Y_PNS,X_A.Point,Y_A.Point,⋯,Y_Gonion,X_Ramus.Point,Y_Ramus.Point,X_Distal.Aspect.of.Condyle,Y_Distal.Aspect.of.Condyle,X_Condylion,Y_Condylion,X_Nasion,Y_Nasion,Patient_ID
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,CT-038,A.I.L,0,0,-29.0,-37.0,16.6,-48.2,66.5,-57.9,⋯,-87.2,-6.7,-78.2,-17.5,-24.3,-11.8,-23.1,71.1,3.6,CT-038-A.I.L
2,CT-050,A.I.L,0,0,-31.4,-33.5,15.1,-41.3,61.7,-47.8,⋯,-85.1,-3.6,-73.8,-16.2,-17.5,-9.2,-14.8,64.9,10.3,CT-050-A.I.L
3,CT-056,A.I.L,0,0,-24.0,-38.4,17.9,-46.7,62.3,-53.2,⋯,-84.6,8.8,-76.1,-14.8,-24.1,-5.8,-20.4,61.2,-0.8,CT-056-A.I.L
4,CT-058,A.I.L,0,0,-18.4,-28.6,18.9,-34.5,57.0,-36.7,⋯,-59.5,-2.5,-53.6,-13.0,-15.8,-5.1,-14.2,56.4,8.2,CT-058-A.I.L
5,CT-288,A.I.L,0,0,-68.3,-94.3,34.6,-69.7,126.0,-83.1,⋯,-163.7,-2.0,-146.0,-44.5,-34.4,-36.0,-33.7,134.6,20.0,CT-288-A.I.L
6,CT-206,A.I.L,0,0,-23.7,-36.7,21.7,-45.9,66.7,-43.9,⋯,-83.6,0.6,-75.0,-17.8,-24.4,-13.7,-21.2,63.9,16.9,CT-206-A.I.L


In [6]:
df_replicate_coordinates %>% select(-Patient, -Operator) %>% column_to_rownames(var = "Patient_ID") -> df_coordinates

In [7]:
# Align coordinates annotations through Generalized Procrustes Analysis (GPA)
gpa <- generalized_procrustes(df_coordinates)

In [8]:
df_aligned_coordinates <- gpa$df_rotated
df_aligned_coordinates$Patient <- df_replicate_coordinates$Patient
df_aligned_coordinates$Operator <- factor(df_replicate_coordinates$Operator)
df_aligned_coordinates_pivot <- df_aligned_coordinates %>%
  pivot_longer(cols = starts_with(c("X_", "Y_")), names_to = "Landmark", values_to = "Value")
head(df_aligned_coordinates_pivot)


Patient,Operator,Landmark,Value
<chr>,<fct>,<chr>,<dbl>
CT-038,A.I.L,X_Sella,-41.31777
CT-038,A.I.L,X_Basion,-73.20422
CT-038,A.I.L,X_PNS,-12.77754
CT-038,A.I.L,X_A.Point,52.96823
CT-038,A.I.L,X_B.Point,66.84070
CT-038,A.I.L,X_Pogonion,73.00775


# Analysis of Variance (ANOVA) 

Assess annotation differences between the two operators

In [9]:
# Evaluate the effect of the operator on the aligned data
model <- aov(Value ~ Landmark * Operator, data = df_aligned_coordinates_pivot)
summary(model)

                   Df  Sum Sq Mean Sq  F value Pr(>F)    
Landmark           23 1328359   57755 2466.240 <2e-16 ***
Operator            1       0       0    0.000  1.000    
Landmark:Operator  23     282      12    0.523  0.968    
Residuals         432   10117      23                    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

# Intraclass Correlation Coefficient

In [10]:
landmark_labels <- colnames(gpa$df_rotated)
df_icc_landmarks <- data.frame() #var = character(), ICC = numeric(), F = numeric(), df1 = numeric(), df2 = numeric(), p = numeric(), lower_bound = numeric(), upper_bound = numeric())

for (var in landmark_labels) {
    icc_data_wide <- df_aligned_coordinates %>% select(Patient, Operator, !!sym(var)) %>%
    pivot_wider(names_from = Operator, values_from = !!sym(var), values_fn = list) %>%
    column_to_rownames(var = "Patient")

    # Compute ICC
    icc_result <- ICC(icc_data_wide)$results %>% filter(type == "ICC3")
    icc_result$var <- var

    df_icc_landmarks <- rbind(df_icc_landmarks, icc_result)
}


# Remove col named 'type'
df_icc_landmarks <- df_icc_landmarks[, -1]

# Change rownames to the col 'var'
rownames(df_icc_landmarks) <- df_icc_landmarks$var
df_icc_landmarks$var <- NULL

# Round the values to 2 decimal places
df_icc_landmarks <- round(df_icc_landmarks, 2)

# Save the ICC results
write.csv(df_icc_landmarks, file = paste0(output_path, "icc_landmarks.csv"), row.names = TRUE)

df_icc_landmarks

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')



,ICC,F,df1,df2,p,lower bound,upper bound
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
X_Sella,0.92,22.81,9,9,0.00,0.70,0.98
Y_Sella,0.81,9.78,9,9,0.00,0.42,0.95
X_Basion,0.74,6.59,9,9,0.00,0.24,0.93
Y_Basion,0.81,9.51,9,9,0.00,0.41,0.95
X_PNS,0.35,2.10,9,9,0.14,-0.32,0.79
Y_PNS,0.96,44.49,9,9,0.00,0.83,0.99
X_A.Point,0.72,6.22,9,9,0.01,0.21,0.92
Y_A.Point,0.88,15.27,9,9,0.00,0.58,0.97
X_B.Point,0.77,7.59,9,9,0.00,0.31,0.94
